##
This notebook will be a 'pivot' since we are downloading Lidar data and not trying to 'burn-in' heights from MS building data.


In [1]:
# import pyproj
# import os
# import rasterio
# import duckdb
# import boto3
# import geopandas as gpd
# import numpy as np
# from botocore import UNSIGNED
# from botocore.config import Config
# from rasterio.windows import from_bounds
# from rasterio.io import MemoryFile
# from rasterio.mask import mask
# from rasterio.session import AWSSession
# from rasterio.warp import calculate_default_transform
# from rasterio.crs import CRS
# from shapely.geometry import box, mapping
# from rasterio.plot import show
# import mercantile as merc
# from matplotlib import pyplot
# from pyarrow import parquet

In [24]:
import pdal, rasterio, os, json, requests, numpy as np
from osgeo import gdal
from pyproj import Transformer
from rasterio.transform import rowcol

In [4]:
# PDAL Pipeline for Obtaining LAZ and converting it to DSM

output_path = os.path.join("data", "stl_mo_dsm.tif")

pipeline_json = {
    "pipeline": [
        {
            "type": "readers.las",
            "filename": "https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/legacy/MO_STLOUIS_2012/LAZ/USGS_LPC_MO_STLOUIS_2012_000054.laz"
        },
        {
            "type": "filters.range",
            "limits": "Classification![7:7]" 
        },
        {
            "type": "filters.outlier",
            "method": "statistical",
            "mean_k": 12,
            "multiplier": 2.0
        },
        {
            "type": "writers.gdal",
            "filename": output_path,
            "nodata": -9999,
            "resolution": 1.0,
            "output_type": "max",
            "override_srs": "EPSG:26915"
        }
    ]
}

pipeline = pdal.Pipeline(json.dumps(pipeline_json))
pipeline.execute()

30936168

In [25]:
url = 'https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/legacy/MO_STLOUIS_2012/LAZ/USGS_LPC_MO_STLOUIS_2012_000054.laz'
local_file = 'data/stl_lidar.laz'

# Download the file
print("Downloading LAZ file... (this may take a minute)")
response = requests.get(url, stream=True)

with open(local_file, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"LAZ file downloaded to {local_file}")

LAZ file downloaded to data/stl_lidar.laz


In [5]:
# Create transformer from WGS84 to EPSG: 26915
transformer = Transformer.from_crs("EPSG:4326", "EPSG:26915", always_xy=True)

# Arch Coordinates (The 'Observer' location)
lon, lat = -90.1848, 38.6247

# Transform
x, y = transformer.transform(lon, lat)

print(f"UTM Coordinates: X={x}, Y={y}")

UTM Coordinates: X=745077.714415889, Y=4278891.007040578


In [6]:
# Open the DSM

with rasterio.open('data/stl_mo_dsm.tif') as src:
    dsm_data = src.read(1)

    transform = src.transform
    crs = src.crs
    profile = src.profile

    print(f"DSM shape: {dsm_data.shape}")
    print(f"Transform: {transform}")

DSM shape: (4500, 4163)
Transform: | 1.00, 0.00, 741500.01|
| 0.00,-1.00, 4281800.00|
| 0.00, 0.00, 1.00|


In [7]:
# Converting arch coordinates to pixel row/column

# Arch center coords
arch_x, arch_y = 745062.40, 4278879.19

# Convert to pixel pos
row, col = rowcol(transform, arch_x, arch_y)

print(f"The Arch is at pixel row={row}, col={col}")

The Arch is at pixel row=2920, col=3562


In [8]:
# Using numpy array to create mask of 100m buffer from point
rows, cols = dsm_data.shape
row_grid, col_grid = np.ogrid[0:rows, 0:cols]

# Calculate distance from each pixel to arch center
distances = np.sqrt((row_grid - row)**2 + (col_grid - col)**2)

# Create mask: True for pixels within 100m
radius = 100
arch_mask = distances <= radius

print(f"Number of pixels in arch footprint: {arch_mask.sum()}")

Number of pixels in arch footprint: 31417


In [9]:
# Cap arch pixels at target height. Modify original dsm

# Create copy of dsm
dsm_modified = dsm_data.copy()

# Target height (halfway up the arch)
target_height = 227.0

# Cap pixels in arch footprint where arch mask == True and elevation > target_height, set to target_height
dsm_modified[arch_mask & (dsm_modified > target_height)] = target_height

print(f"Modified DSM - pixels changed: {np.sum((dsm_data != dsm_modified) & arch_mask)}")


Modified DSM - pixels changed: 1503


In [10]:
# Update the profile for the output file
output_profile = profile.copy()

# Write the modified DSM
output_filename = 'data/stl_mo_dsm_half.tif'

with rasterio.open(output_filename, 'w', **output_profile) as dst:
    dst.write(dsm_modified, 1)

print(f"Modified DSM saved to {output_filename}")

CPLE_AppDefinedError: Deleting data/stl_mo_dsm_half.tif failed: Permission denied

In [11]:
print(f"{arch_x}, {arch_y}")

745062.4, 4278879.19


In [19]:
# Run viewshed analysis

ds = gdal.Open('data/stl_mo_dsm.tif')
band = ds.GetRasterBand(1)

# Run viewshed
viewshed_result = gdal.ViewshedGenerate(
    band,
    'GTiff',
    'data/viewshed_2m.tif',
    [],
    arch_x,
    arch_y,
    2,  # observer height in meters
    1.7,
    255,
    0,
    0,
    -9999,
    0.85714,
    gdal.GVM_Edge,
    0   
)
